# 13 - Model Context Protocol (MCP)

## Scenario: Northstar Log VPC

Northstar's main agent orchestrator runs in a DMZ, but the checkout logs live in a highly secure VPC. Instead of opening firewall holes and embedding database credentials into the main agent codebase, Northstar deploys an **MCP Server** inside the secure VPC. 

The Model Context Protocol (MCP) provides a standardized way for AI applications (like our agent) to connect to external data sources and tools, regardless of what language or framework they are built in.

In [ ]:
import json
# In this module we use the official `mcp` Python SDK
import mcp.types as types
from mcp.server import Server
from mcp.server.stdio import stdio_server

# 1. Initialize the MCP Server
# This represents the secure server running inside the VPC.
app = Server("northstar-log-server")

@app.list_tools()
async def list_tools() -> list[types.Tool]:
    """
    When an MCP client connects, it asks the server: 'What tools do you have?'
    The server responds with standard JSON Schema tool definitions.
    """
    return [
        types.Tool(
            name="query_checkout_logs",
            description="Securely queries the internal checkout logs database.",
            inputSchema={
                "type": "object",
                "properties": {
                    "region": {"type": "string"},
                    "error_code": {"type": "string"}
                },
                "required": ["region"]
            }
        )
    ]

@app.call_tool()
async def call_tool(name: str, arguments: dict) -> list[types.TextContent]:
    """
    When the client wants to execute a tool, it sends the request to the MCP server.
    The server executes the sensitive code locally and returns the text result.
    """
    if name == "query_checkout_logs":
        region = arguments.get("region")
        error_code = arguments.get("error_code", "any")
        
        # Simulated secure DB query
        print(f"[Secure VPC] Executing log query for {region} (Error: {error_code})")
        
        if region == "eu-west":
            result = f"Found 312 errors matching {error_code} in {region}. Most common: VAT Validation Timeout."
        else:
            result = f"No significant errors found in {region}."
            
        return [types.TextContent(type="text", text=result)]
    
    raise ValueError(f"Unknown tool: {name}")


## 1. Why use MCP?

Without MCP, every agent framework (LangGraph, AutoGen, CrewAI) requires you to write custom integration code to talk to external systems. If you write a tool for AutoGen, it doesn't work in CrewAI. 

With MCP:
1. You write the tool **once** as an MCP Server.
2. Any MCP-compatible Client (Claude Desktop, your custom LangGraph agent, Cursor) can instantly discover and use the tool via a standard protocol (stdio or SSE).
3. The Server retains complete control over authentication, rate limiting, and execution.

## 2. Connecting a Client

In a real production environment, you would use `mcp.client` to connect to this server over HTTP Server-Sent Events (SSE) or Standard IO. Let's simulate what a client does when it connects.

In [ ]:
# Simulation of an MCP Client interacting with the MCP Server
import asyncio

async def simulate_mcp_client():
    print("--- MCP Client Connecting to Server ---")
    
    # 1. Client asks for available tools
    available_tools = await list_tools()
    print("\n[Client] Discovered Tools:")
    for tool in available_tools:
        print(f" - {tool.name}: {tool.description}")
        
    # 2. Client decides to call a tool (usually dictated by an LLM)
    print("\n[Client] Calling 'query_checkout_logs' via MCP...")
    response = await call_tool("query_checkout_logs", {"region": "eu-west", "error_code": "500"})
    
    print("\n[Client] Received Response from Server:")
    print(response[0].text)

# Run the simulation
# (We use asyncio.run in a real script, but in Jupyter we can just await it)
await simulate_mcp_client()


## Watch For

- **Latency**: Using HTTP SSE for MCP means every tool call incurs network latency. Don't use MCP for extremely fast, local string manipulation tools.
- **Security Illusion**: Just because a tool is in an MCP server doesn't mean it's secure. The server still needs strict auth and Pydantic validation (as covered in previous modules).
- **Transport Limitations**: `stdio` transport is great for local sub-processes (like Claude Desktop running local scripts), but SSE is required for remote VPC servers.

## Checkpoint

**1. What problem does the Model Context Protocol (MCP) solve?**
- A) It trains LLMs faster.
- B) It standardizes how AI applications connect to and discover external tools/data sources, decoupling the agent from the tool implementation.
- C) It replaces JSON schema.
- D) It prevents LLM hallucinations.

**2. In an MCP architecture, who actually executes the tool's backend logic?**
- A) The LLM provider (e.g. Anthropic).
- B) The MCP Client.
- C) The MCP Server.
- D) The End User.
